# Cue (Li+24) mock-generation tests (T1–T5)

Goal: validate the Cue nebular emulator (`NebStepBasis` + `cue_stellar_nebular`) as a free-N/O replacement for the fixed-N/O FSPS+Byler grid, **before** committing to a 500k regen.

Cue free nebular params (ionizing spectrum tied to FSPS stars, `use_stellar_ionizing=True`):
`gas_logz`, `gas_logu`, `gas_lognH`, **`gas_logno` [N/O]**, `gas_logco` [C/O].

**Prediction to keep in mind:** Cue should absorb the ~292 high-N/O outliers, NOT the ~60 quiescent (those are an SFH-coverage gap).

Reuses `make_prospector_model_sed.py` for shared pieces (priors, stochastic agebins, DESI_WAV, R_mat, Spectrum obs). Swaps: `FastStepBasis`→`NebStepBasis`, FSPS `nebular`→`cue_stellar_nebular`.

## T1 — Environment / deps
Confirm `cuejax` is installed and `NebStepBasis`/`Emulator` load (the warmup compile happens here, can take ~10-30s).

In [ ]:
import copy, time, importlib.util
from pathlib import Path
import numpy as np
from prospect.models import priors, transforms
from prospect.models.templates import TemplateLibrary, adjust_stochastic_params
from prospect.models.sedmodel import HyperSpecModel

REPO = Path.cwd().parents[0] if Path.cwd().name == "tmp" else Path.cwd()
spec = importlib.util.spec_from_file_location(
    "msed", REPO / "bin" / "model_seds" / "make_prospector_model_sed.py"
)
msed = importlib.util.module_from_spec(spec)
spec.loader.exec_module(msed)
pr = msed.priors_dict
DESI_WAV = np.asarray(msed.DESI_WAV)
n_wave = DESI_WAV.size

import cuejax

print("cuejax", getattr(cuejax, "__version__", "?"))
from prospect.sources import NebStepBasis

t0 = time.time()
sps = NebStepBasis()
print(f"NebStepBasis built in {time.time() - t0:.1f}s")
print("n Cue lines:", np.asarray(sps.emline_wavelengths).size)

## Cue parset builder
Same stochastic SFH + C&F dust + dust_emission as the FSPS generator, but `cue_stellar_nebular` nebular block and the 3 new abundance params. `nebemlineinspec=False` + Spectrum obs (lines via SpecModel eline path, honoring `eline_sigma`).

NOTE: `gas_lognH`/`gas_logno`/`gas_logco` are NOT in the current priors npz — set to defaults here; sampling them must be added to `get_stochastic_priors.py` before a full regen.

In [ ]:
def build_cue_base():
    t = copy.deepcopy(TemplateLibrary["stochastic_sfh"])
    t.update(copy.deepcopy(TemplateLibrary["dust_emission"]))
    t.update(copy.deepcopy(TemplateLibrary["cue_stellar_nebular"]))
    t["nebemlineinspec"] = {"N": 1, "isfree": False, "init": False}
    t["dust_type"]["init"] = 0
    t["mass"]["init"] = 10**10.7
    t["dust1"] = {
        "N": 1,
        "isfree": False,
        "depends_on": transforms.dustratio_to_dust1,
        "init": 0.0,
    }
    t["dust_ratio"] = {
        "N": 1,
        "isfree": True,
        "init": 1.0,
        "prior": priors.ClippedNormal(mini=0.0, maxi=2.0, mean=1.0, sigma=0.3),
    }
    t["dust_index"] = {
        "N": 1,
        "isfree": True,
        "init": 0.0,
        "prior": priors.TopHat(mini=-1.0, maxi=0.4),
    }
    t["sigma_smooth"] = {"N": 1, "isfree": False, "init": 200.0}
    t["smoothtype"] = {"N": 1, "isfree": False, "init": "vel"}
    t["fftsmooth"] = {"N": 1, "isfree": False, "init": True}
    t["eline_sigma"] = {"N": 1, "isfree": False, "init": 100.0}
    return t


def build_cue_parset(i, base, gas_logno=0.0, gas_logco=0.0, gas_lognH=2.0):
    t = copy.deepcopy(base)
    t["logmass"]["init"] = pr["stellar_masses"][i]
    t["logzsol"]["init"] = pr["stellar_metallicities"][i]
    t["dust_index"]["init"] = pr["ns"][i]
    t["dust_ratio"]["init"] = pr["tau_dust_1s"][i]
    t["dust2"]["init"] = pr["tau_dust_2s"][i]
    t["duste_umin"]["init"] = pr["u_mins"][i]
    t["duste_qpah"]["init"] = pr["q_pahs"][i]
    t["duste_gamma"]["init"] = pr["gamma_es"][i]
    t["gas_logz"]["init"] = pr["gas_metallicities"][i]
    t["gas_logu"]["init"] = pr["gas_ionization_parameters"][i]
    t["gas_lognH"]["init"] = gas_lognH
    t["gas_logno"]["init"] = gas_logno
    t["gas_logco"]["init"] = gas_logco
    z = pr["redshifts"][i]
    t["zred"]["init"] = z
    t["agebins"]["init"] = msed.make_stochastic_agebins(z=z)
    t["sigma_reg"]["init"] = pr["sigma_regs"][i]
    t["tau_eq"]["init"] = pr["tau_eqs"][i]
    t["tau_in"]["init"] = pr["tau_ins"][i]
    t["sigma_dyn"]["init"] = pr["sigma_dyns"][i]
    t["tau_dyn"]["init"] = pr["tau_dyns"][i]
    t["sigma_smooth"]["init"] = pr["sigma_smooths"][i]
    t["eline_sigma"]["init"] = pr["sigma_gass"][i]
    t = adjust_stochastic_params(t)
    t["logsfr_ratios"]["init"] = t["logsfr_ratios"]["prior"].sample()
    return t


obs = msed._make_obs()


def predict_cue(t):
    m = HyperSpecModel(configuration=t)
    preds, _ = m.predict(m.theta, [obs], sps=sps)
    return np.asarray(preds[0]), m


BASE = build_cue_base()
ew_ref = np.asarray(sps.emline_wavelengths)


def lidx(lam):
    return int(np.argmin(np.abs(ew_ref - lam)))


LI = {
    k: lidx(v)
    for k, v in dict(
        Hb=4862.7, OIII=5008.2, Ha=6564.6, NII=6585.3, SII1=6718.3, SII2=6732.7
    ).items()
}
print("Cue line matches:", {k: round(float(ew_ref[i]), 2) for k, i in LI.items()})

## T2 — single-spectrum sanity
Pick a low-z, high-ionization sample. Check: emission peaks present, intrinsic (dust-off) Hα/Hβ ≈ 2.86, BPT ratios sane.

In [ ]:
cand = np.where((pr["gas_ionization_parameters"] > -1.5) & (pr["redshifts"] < 0.1))[0][
    0
]
t = build_cue_parset(int(cand), BASE)
sp, m = predict_cue(t)
z = float(pr["redshifts"][cand])
xr = DESI_WAV / (1 + z)


def pc(rest):
    j = int(np.argmin(np.abs(xr - rest)))
    c = np.median(np.r_[sp[j - 25 : j - 10], sp[j + 10 : j + 25]])
    return round(float(sp[j - 3 : j + 4].max() / c), 2) if c else np.nan


print(
    "idx",
    int(cand),
    "z",
    round(z, 3),
    "peak/cont  Hb",
    pc(4862.68),
    "OIII",
    pc(5008.24),
    "Ha",
    pc(6564.61),
)
el = np.asarray(m._eline_lum)
print(
    "intrinsic-ish Cue Ha/Hb (attenuated lum):",
    round(float(el[LI["Ha"]] / el[LI["Hb"]]), 3),
)
# dust-off decrement
t0 = build_cue_parset(int(cand), BASE)
t0["dust2"]["init"] = 0.0
t0["dust_ratio"]["init"] = 0.0
_, m0 = predict_cue(t0)
el0 = np.asarray(m0._eline_lum)
print(
    "dust-off Cue Ha/Hb:",
    round(float(el0[LI["Ha"]] / el0[LI["Hb"]]), 3),
    " (expect ~2.8-3.0)",
)
print(
    "BPT (dust-off):  log[NII]/Ha",
    round(float(np.log10(el0[LI["NII"]] / el0[LI["Ha"]])), 2),
    " log[OIII]/Hb",
    round(float(np.log10(el0[LI["OIII"]] / el0[LI["Hb"]])), 2),
)

## T3 — N/O lever (KEY TEST)
Vary `gas_logno` at fixed everything else. [NII]/Hα and [NII]/[SII] MUST increase with `gas_logno`. If flat → the knob isn't doing what we need.

In [ ]:
idx = int(cand)
print(
    f"{'gas_logno':>9} {'log[NII]/Ha':>11} {'log[NII]/[SII]':>14} {'log[OIII]/Hb':>12}"
)
for no in [-1.0, -0.5, 0.0, 0.3, 0.7]:
    t = build_cue_parset(idx, BASE, gas_logno=no)
    t["dust2"]["init"] = 0.0
    t["dust_ratio"]["init"] = 0.0
    _, mm = predict_cue(t)
    e = np.asarray(mm._eline_lum)
    nii_ha = np.log10(e[LI["NII"]] / e[LI["Ha"]])
    nii_sii = np.log10(e[LI["NII"]] / (e[LI["SII1"]] + e[LI["SII2"]]))
    oiii_hb = np.log10(e[LI["OIII"]] / e[LI["Hb"]])
    print(f"{no:>9.2f} {nii_ha:>11.3f} {nii_sii:>14.3f} {oiii_hb:>12.3f}")

## T4 — timing
Per-spectrum cost (after warmup) → estimate 500k feasibility vs FSPS.

In [ ]:
import time

idxs = np.where(pr["redshifts"] < 0.4)[0][:10]
_ = predict_cue(build_cue_parset(int(idxs[0]), BASE))  # warm
t0 = time.time()
for i in idxs:
    predict_cue(build_cue_parset(int(i), BASE))
dt = (time.time() - t0) / len(idxs)
print(
    f"Cue per-spectrum: {dt:.3f}s  -> 500k single-core ~{dt * 500000 / 3600:.1f} h  (8 workers ~{dt * 500000 / 3600 / 8:.1f} h)"
)

## T5 — locus coverage
Generate a small Cue batch with `gas_logno` sampled uniformly (its prior range), compute BPT/[NII]/[SII] ratios from `_eline_lum`, and check the locus reaches the high-N/O region the 292 outliers occupy (FSPS mock [NII]/[SII] maxed ~0.28; outliers spilled beyond).

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
N = 300
sel = rng.choice(np.where(pr["redshifts"] < 0.4)[0], N, replace=False)
rows = []
for i in sel:
    no = rng.uniform(-1.0, np.log10(5.4))
    co = rng.uniform(-1.0, np.log10(5.4))
    nh = rng.uniform(1.0, 4.0)
    t = build_cue_parset(int(i), BASE, gas_logno=no, gas_logco=co, gas_lognH=nh)
    t["dust2"]["init"] = 0.0
    t["dust_ratio"]["init"] = 0.0  # intrinsic ratios for locus
    _, mm = predict_cue(t)
    e = np.asarray(mm._eline_lum)
    rows.append(
        [
            np.log10(e[LI["NII"]] / e[LI["Ha"]]),
            np.log10(e[LI["OIII"]] / e[LI["Hb"]]),
            np.log10((e[LI["SII1"]] + e[LI["SII2"]]) / e[LI["Ha"]]),
            np.log10(e[LI["NII"]] / (e[LI["SII1"]] + e[LI["SII2"]])),
        ]
    )
R = np.array(rows)
R = R[np.all(np.isfinite(R), axis=1)]
print("Cue locus ranges (p1,p99):")
for k, name in enumerate(["[NII]/Ha", "[OIII]/Hb", "[SII]/Ha", "[NII]/[SII]"]):
    print(
        f"  {name:11s}: [{np.percentile(R[:, k], 1):.2f}, {np.percentile(R[:, k], 99):.2f}]"
    )
print("\nFSPS mock [NII]/[SII] range was [-0.77, 0.28]; outliers spilled beyond 0.28.")
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(R[:, 0], R[:, 1], s=8, alpha=0.5)
ax[0].set_xlabel("log[NII]/Ha")
ax[0].set_ylabel("log[OIII]/Hb")
ax[0].set_title("Cue NII-BPT locus")
ax[1].scatter(R[:, 3], R[:, 1], s=8, alpha=0.5)
ax[1].axvline(0.28, color="r", ls="--", label="FSPS max")
ax[1].set_xlabel("log[NII]/[SII]")
ax[1].set_ylabel("log[OIII]/Hb")
ax[1].legend()
ax[1].set_title("N/O reach")
fig.tight_layout()
plt.show()

## Results / notes

- **T1 (env):** PASS. cuejax loads, `NebStepBasis` builds, Cue line list maps correctly (Hb 4862.76, OIII 5008.31, Ha 6564.72, NII 6585.37, SII 6718.40/6732.78).
- **T2 (sanity):** PASS. Lines present; dust-off Ha/Hb = 2.94 (Case B, matches FSPS-Byler 2.99); BPT ratios sane.
- **T3 (N/O lever, KEY):** PASS. [NII]/Ha and [NII]/[SII] climb monotonically with `gas_logno` (+1.3 / +1.7 dex over [-1, 0.7]); [NII]/[SII] reaches +0.95 (single object). [OIII]/Hb co-varies (drops at high N/O = N cooling) -> gas_logno not orthogonal to ionization, expected.
- **T4 (timing):** 1.83 s/spectrum was JAX **compile** overhead (small sample). Steady-state (from T5, 300 spectra/11.6s) = **0.039 s/spectrum** -> 500k ~5.4 h single-core, **~40-60 min on 8 workers**. Full regen is cheap. Watch: JAX+multiprocessing must use spawn (not fork) to avoid deadlock.
- **T5 (locus coverage):** PASS. Sampled Cue population reaches [NII]/[SII] = +0.83 vs FSPS-Byler ceiling +0.28 -> covers the high-N/O region the 292 outliers occupy. Locus wider than DESI in places (uniform abundance priors) -> fine for SBI coverage.

**Verdict:** Cue validated in principle. Next = T6: small validation regen (~50k, ~3 h) -> noise -> latents -> outliers. Prediction: 352 -> ~60 (N/O absorbed, ~60 quiescent remain, since Cue does NOT fix the SFH-quiescence gap).

**Before T6:** add `gas_lognH`/`gas_logno`/`gas_logco` sampling to `get_stochastic_priors.py`; write the Cue variant of `make_prospector_model_sed.py` (swap sps->NebStepBasis, template->cue_stellar_nebular, store Cue `line_wave`).


In [ ]:
import importlib.util

s = importlib.util.spec_from_file_location(
    "mcue", REPO / "bin" / "model_seds" / "make_cue_model_sed.py"
)
mcue = importlib.util.module_from_spec(s)
s.loader.exec_module(mcue)  # loads cue priors npz
print(
    "n_spectra",
    mcue.n_spectra,
    "has cue keys:",
    all(k in mcue.priors_dict for k in ["gas_lognHs", "gas_lognos", "gas_logcos"]),
)
start, stop, block, ratios, lum = mcue.worker_block(0, 5)  # single-process, 5 spectra
print("block", block.shape, "lum", lum.shape, "line_wave", mcue._get_line_wave().shape)
# lines present?
wv = np.asarray(mcue.DESI_WAV)
z0 = mcue.priors_dict["redshifts"][0]
xr = wv / (1 + z0)
j = int(np.argmin(np.abs(xr - 6564.6)))
print(
    "Ha peak/cont",
    round(
        float(
            block[0, j - 3 : j + 4].max()
            / np.median(np.r_[block[0, j - 25 : j - 10], block[0, j + 10 : j + 25]])
        ),
        2,
    ),
)